In [2]:
from cmlreaders import CMLReader, get_data_index
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle
import warnings
from pathlib import Path
from datetime import datetime
from scipy import stats
import subprocess
import sys
import os
from pathlib import Path
warnings.filterwarnings('ignore')
from irasa.IRASA import IRASA, SSL_transform
import cmlreaders as cml
from irasa.IRASA import IRASA
# Load index of all experiments
df = get_data_index()

# Filter for TH experiments

EXPERIMENT='TH1'
whole_df = cml.CMLReader.get_data_index()
whole_df['experiment'].unique()

array(['ValueCourier', 'ltpFR', 'ltpFR2', 'VFFR', 'ltpRepFR',
       'NiclsCourierClosedLoop', 'NiclsCourierReadOnly',
       'ltpDelayRepFRReadOnly', 'CourierReinstate1', 'VCBehOnly',
       'ltpDBOY1', 'prelim', 'EFRCourierOpenLoop', 'EFRCourierReadOnly',
       'FR1', 'FR2', 'PAL1', 'YC1', 'PAL2', 'catFR1', 'YC2', 'catFR2',
       'PS1', 'ICatFR1', 'ICatFR6', 'IFR1', 'IFR6', 'PS3', 'PS2', 'TH1',
       'FR3', 'PS2.1', 'PAL3', 'TH3', 'OPS', 'RepFR1', 'catFR3', 'FR5',
       'PS4_catFR', 'THR', 'PS4_FR', 'PAL5', 'THR1', 'catFR5',
       'PS4_catFR5', 'FR6', 'PS5_catFR', 'catFR6', 'TICL_FR',
       'LocationSearch', 'TICL_catFR', 'DBOY1', 'CatFR6', 'RepFR2', 'CPS',
       'pyFR'], dtype=object)

In [16]:
EXPERIMENT='YC1'
whole_df = cml.CMLReader.get_data_index()
exp_df   = whole_df[whole_df['experiment'] == EXPERIMENT]

SUBJECTS = sorted(exp_df['subject'].unique())
SUBJECTS

['R1001P',
 'R1006P',
 'R1008J',
 'R1009W',
 'R1010J',
 'R1013E',
 'R1014D',
 'R1015J',
 'R1018P',
 'R1019J',
 'R1021D',
 'R1023J',
 'R1024E',
 'R1025P',
 'R1026D',
 'R1030J',
 'R1032D',
 'R1033D',
 'R1034D',
 'R1037D',
 'R1041M',
 'R1042M',
 'R1044J',
 'R1045E',
 'R1047D',
 'R1048E',
 'R1049J',
 'R1050M',
 'R1051J',
 'R1052E',
 'R1054J',
 'R1056M',
 'R1059J',
 'R1060M',
 'R1061T',
 'R1062J',
 'R1063C',
 'R1064E',
 'R1065J',
 'R1066P',
 'R1067P',
 'R1068J',
 'R1069M',
 'R1073J',
 'R1074M',
 'R1075J',
 'R1077T',
 'R1079E',
 'R1083J',
 'R1086M',
 'R1089P',
 'R1096E',
 'R1101T',
 'R1106M',
 'R1112M',
 'R1114C',
 'R1124J',
 'R1177M']

In [17]:
from cmlreaders import CMLReader

subject ='R1006P'
experiment = EXPERIMENT   # override if needed
session = 0

print(subject, experiment, session)

reader = CMLReader(
    subject=subject,
    experiment=experiment,
    session=session
)
reader.load('pairs')
events = reader.load('events')
events['type'].unique()
df=events[events['type']=='REC_EVENT']
events['type'].unique()

R1006P YC1 0


array(['NAV_PRACTICE_LEARN', 'NAV_PRACTICE_TEST', 'NAV_LEARN', 'NAV_TEST'],
      dtype=object)

In [18]:
list(events)

['eegoffset',
 'block',
 'block_num',
 'eegfile',
 'env_size',
 'exp_version',
 'experiment',
 'is_stim',
 'montage',
 'msoffset',
 'mstime',
 'obj_locs',
 'paired_block',
 'path',
 'protocol',
 'recalled',
 'resp_dist_err',
 'resp_locs',
 'resp_path_length',
 'resp_performance_factor',
 'resp_reaction_time',
 'resp_travel_time',
 'session',
 'start_locs',
 'stim_params',
 'stimulus',
 'stimulus_num',
 'subject',
 'type']

In [73]:
events[['type','eegoffset','item_name','confidence','recalled']]

,type,eegoffset,item_name,confidence,recalled
0,CHEST,49480,water bottle,2,False
1,CHEST,53927,battery,2,True
2,CHEST,58776,,-999,False
3,CHEST,63543,megaphone,2,True
4,REC,74393,battery,2,True
...,...,...,...,...,...
255,CHEST,2721223,beachball,2,False
256,CHEST,2725338,cello,2,True
257,REC,2735488,cello,2,True
258,REC,2742403,beachball,2,False


In [78]:
events['mstime_diff'] = None
for item in events['item_name'].unique():
    chest_time = events[(events['type']=='CHEST') & 
                        (events['item_name']==item)]['eegoffset'].values
    rec_time = events[(events['type']=='REC') & 
                      (events['item_name']==item)]['eegoffset'].values
    print(item, rec_time - chest_time)

water bottle [35111]
battery [20466]


ValueError: operands could not be broadcast together with shapes (0,) (60,) 

In [79]:
rec_events = events[events['type']=='REC']
print(rec_events[['trial', 'item_name', 'reactionTime', 'eegoffset', 'confidence', 'recalled']].head(20))
print("\nReaction Time Stats:")
print(rec_events['reactionTime'].describe())

    trial      item_name  reactionTime  eegoffset  confidence  recalled
4       0        battery          9866      74393           2      True
5       0   water bottle          6850      84591           2     False
6       0      megaphone          4917      91741           2      True
11      1         anchor         19550     143988           1     False
12      1          slide         16333     163854           1     False
13      1          yacht          9950     180536           1     False
18      2       push pin         13584     238166           2      True
19      2     binoculars          7784     252065           2      True
24      3   refrigerator         11816     305964           2      True
25      3           oven          5783     318079           2     False
30      4      lightbulb          7267     367109           2      True
31      4      hourglass          5084     374759           2      True
32      4        balloon          9517     380126           1   

In [77]:
reader = cml.CMLReader(subject, experiment, session)
eeg = reader.load('eeg')
print(eeg.samplerate)

1000


In [60]:
df[df['vocalization']==0]['intrusion']

KeyError: 'vocalization'

In [ ]:
from cmlreaders import CMLReader, get_data_index

# Step 1: verify existence
df = get_data_index()

valid = df[
    (df.subject == subject) &
    (df.experiment == experiment) &
    (df.session == session)
]

print(valid)

# Step 2: create reader
reader = CMLReader(
    subject=subject,
    experiment=experiment,
    session=session
)

pairs_df = reader.load('pairs')
# Step 4: load
events = reader.load('events')

In [98]:
events=reader.load('events')

FileNotFoundError: Unable to find the requested file in any of the expected locations:
 /protocols/ltp/subjects/LTP564/experiments/PS1/sessions/1/behavioral/current_processed/task_events.json
/data/events/pyFR/LTP564_None_events.mat

In [99]:
events['type'].unique()

array(['store mappings', 'TRIAL_START', 'WORD', 'POINTER_ON',
       'REINSTATEMENT', 'TRIAL_END', 'REC_START', 'REC_WORD', 'REC_STOP',
       'EFR_MARK', 'BREAK_START', 'BREAK_STOP', 'VIDEO_START',
       'VIDEO_STOP', 'MUSIC_VIDEOS_STOP', 'REC_WORD_VV'], dtype=object)

In [17]:
events[events['type']=='REC']['item_name']

4              hammer
5          television
6              jewels
11           trashcan
12              mixer
            ...      
251    digital camera
252               fan
257             yacht
258     chocolate bar
259             stool
Name: item_name, Length: 100, dtype: object

In [75]:
import numpy as np
import pandas as pd
import cmlreaders as cml

# ============================================================
# CORE FUNCTION (your validated version)
# ============================================================

def process_trial(trial_evs):

    enc = trial_evs[trial_evs['type'] == 'CHEST']
    rec = trial_evs[trial_evs['type'] == 'REC']

    if len(enc) < 2 or len(rec) < 2:
        return None

    # Encoding coordinates
    presented = np.column_stack([
        enc['locationX'].values,
        enc['locationY'].values
    ])
    scale = np.std(presented)
    threshold = scale * 0.5

    # Recall coordinates
    recalled = np.column_stack([
        rec['chosenLocationX'].values,
        rec['chosenLocationY'].values
    ])

    # Valid recall filter
    valid_mask = rec['recalled'] == True
    if valid_mask.sum() < 2:
        return None
    recalled = recalled[valid_mask]

    # Map recall → item number
    rec_itemnos = []
    for rc in recalled:
        dists = np.linalg.norm(presented - rc, axis=1)
        idx = np.argmin(dists)
        rec_itemnos.append(idx + 1 if dists[idx] < threshold else -1)
    rec_itemnos = np.array(rec_itemnos)

    # Remove invalid and duplicates
    clean = []
    seen = set()
    for r in rec_itemnos:
        if r > 0 and r not in seen:
            clean.append(r)
            seen.add(r)
    if len(clean) < 2:
        return None
    clean = np.array(clean)

    # Distance matrices over ALL encoded items
    n_items = len(presented)
    spatial_mat = np.linalg.norm(
        presented[:, None, :] - presented[None, :, :], axis=2
    )
    temporal_mat = np.abs(
        np.subtract.outer(enc['chestNum'].values, enc['chestNum'].values)
    )

    # Clustering score
    def compute_clustering(seq, dist_mat):
        scores = []
        seen_inner = set()
        all_items = set(range(1, n_items + 1))

        for i in range(len(seq) - 1):   # <-- MUST be len(seq)-1, not len(seq)
            curr = seq[i]
            nxt  = seq[i + 1]
            seen_inner.add(curr)

            remaining = all_items - seen_inner  # all encoded items not yet visited
            if len(remaining) < 1:
                continue

            actual   = dist_mat[curr - 1, nxt - 1]
            possibles = [dist_mat[curr - 1, j - 1] for j in remaining]

            rank  = np.sum(np.array(possibles) > actual)
            score = rank / len(possibles)
            scores.append(score)

        return np.array(scores)

    sp = compute_clustering(clean, spatial_mat)
    tp = compute_clustering(clean, temporal_mat)

    return clean, sp, tp
    # ========================
    # Clustering
    # ========================
    def compute_clustering(seq, dist_mat, n_items):
   
        scores = []
        seen = set()
        all_items = set(range(1, n_items + 1))

        for i in range(len(seq) - 1):
            curr = seq[i]
            nxt  = seq[i + 1]
            seen.add(curr)

            # Pool = all encoded items not yet recalled (including nxt)
            remaining = all_items - seen

            if len(remaining) < 1:
                continue

            actual = dist_mat[curr - 1, nxt - 1]
            possibles = [dist_mat[curr - 1, j - 1] for j in remaining]

            rank = np.sum(np.array(possibles) > actual)
            score = rank / len(possibles)   # proportion farther than actual

            scores.append(score)

        return np.array(scores)

# ============================================================
# MAIN PIPELINE
# ============================================================

def run_th_clustering(exp='TH3', save_path='TH_clustering_results.csv'):

    df = cml.get_data_index()

    subjects = df[df['experiment'] == exp]['subject'].unique()

    print(f"Found {len(subjects)} subjects")

    all_rows = []

    for subj in subjects:
        print(f"\nProcessing subject: {subj}")

        sessions = df[
            (df.subject == subj) &
            (df.experiment == exp)
        ].session.unique()

        for sess in sessions:
            print(f"  Session: {sess}")

            try:
                reader = cml.CMLReader(subj, exp, session=sess)
                evs = reader.load('events')
            except Exception as e:
                print(f"  Failed to load: {e}")
                continue

            trials = evs['trial'].unique()

            for trial in trials:

                if trial < 0:
                    continue

                trial_evs = evs[evs['trial'] == trial]

                result = process_trial(trial_evs)

                if result is None:
                    continue

                rec_itemnos, sp, tp = result

                # ========================
                # Save per recall
                # ========================
                for i in range(len(rec_itemnos)):

                    all_rows.append({
                        "subject": subj,
                        "session": sess,
                        "trial": trial,
                        "rec_itemno": rec_itemnos[i],

                        "SP_from_prev": sp[i-1] if i > 0 and i-1 < len(sp) else np.nan,
                        "SP_to_next": sp[i] if i < len(sp) else np.nan,

                        "TP_from_prev": tp[i-1] if i > 0 and i-1 < len(tp) else np.nan,
                        "TP_to_next": tp[i] if i < len(tp) else np.nan,
                    })

    # ============================================================
    # SAVE OUTPUT
    # ============================================================

    if len(all_rows) == 0:
        print("\nWARNING: No data collected")
        return None

    out_df = pd.DataFrame(all_rows)

    out_df.to_csv(save_path, index=False)

    print("\nDONE")
    print(f"Saved: {save_path}")
    print(f"Rows: {len(out_df)}")

    return out_df


# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    run_th_clustering()

Found 5 subjects

Processing subject: R1201P
  Session: 0
  Session: 1

Processing subject: R1208C
  Session: 0

Processing subject: R1237C
  Session: 0
  Session: 1
  Session: 2

Processing subject: R1276D
  Session: 0
  Session: 1
  Session: 2
  Session: 3
  Session: 5

Processing subject: R1289C
  Session: 3

DONE
Saved: TH_clustering_results.csv
Rows: 307


In [45]:
evs.to_csv('example.csv')

In [40]:
print(f"  n_items={len(presented)}, clean={clean}, len(sp)={len(sp)}, len(tp)={len(tp)}")
return clean, sp, tp

NameError: name 'presented' is not defined

In [28]:
trial_evs

,eegoffset,block,chestNum,chosenLocationX,chosenLocationY,confidence,distErr,eegfile,exp_version,experiment,...,reactionTime,recStartLocationX,recStartLocationY,recalled,session,stim_list,stim_params,subject,trial,type
234,1069321,4,1,-999.0000,-999.0000,-999,-999.000000,R1289C_TH1_5_27Mar17_1107,,TH1,...,-999,-999.000,-999,True,5,False,[],R1289C,36,CHEST
235,1071296,4,2,366.9945,344.8759,0,10.622958,R1289C_TH1_5_27Mar17_1107,,TH1,...,8550,384.587,433,True,5,False,[],R1289C,36,CHEST
236,1073671,4,3,401.9486,360.4178,0,6.074910,R1289C_TH1_5_27Mar17_1107,,TH1,...,5451,384.587,433,True,5,False,[],R1289C,36,CHEST
237,1076362,4,4,367.0030,389.8893,2,5.469322,R1289C_TH1_5_27Mar17_1107,,TH1,...,6616,384.587,433,True,5,False,[],R1289C,36,CHEST
238,1081061,4,3,401.9486,360.4178,0,6.074910,R1289C_TH1_5_27Mar17_1107,,TH1,...,5451,384.587,433,True,5,False,[],R1289C,36,REC
239,1083953,4,4,367.0030,389.8893,2,5.469322,R1289C_TH1_5_27Mar17_1107,,TH1,...,6616,384.587,433,True,5,False,[],R1289C,36,REC
240,1087470,4,2,366.9945,344.8759,0,10.622958,R1289C_TH1_5_27Mar17_1107,,TH1,...,8550,384.587,433,True,5,False,[],R1289C,36,REC


In [32]:
list(trial_evs)

['eegoffset',
 'block',
 'chestNum',
 'chosenLocationX',
 'chosenLocationY',
 'confidence',
 'distErr',
 'eegfile',
 'exp_version',
 'experiment',
 'isRecFromNearSide',
 'isRecFromStartSide',
 'is_stim',
 'item_name',
 'listLength',
 'locationX',
 'locationY',
 'montage',
 'msoffset',
 'mstime',
 'navStartLocationX',
 'navStartLocationY',
 'normErr',
 'protocol',
 'radius_size',
 'reactionTime',
 'recStartLocationX',
 'recStartLocationY',
 'recalled',
 'session',
 'stim_list',
 'stim_params',
 'subject',
 'trial',
 'type']